# M0.1 环境、先修自检与失效地图

> **M0 · 起点 / T0 必修**：从 Jupyter 到第一张温度场云图

本 Notebook 按 `slides/M0_1_演示网页_v2.html` 的学生页面重排，配套课程大纲 v3。请从上到下运行代码单元；每个实验代码都包含自己的导入，复制到本地 Jupyter 也能运行。

**课程主线**：先修自测 → 失效演示 → F1–F5 失效地图 → 解析解基准 → 环境自检 → POE 预测/运行/解释 → 改参数 → 向量化对照。

课前完成环境安装与自测；课中完成失效演示、解析云图和诊断；课后补完清单。网页中的 Pyodide 只是零门槛备选，本 Notebook 以本地 Python 3.12 环境为准。

## 1. 学习目标与先修准备

学完本单元，你应能：

1. 说出商业 CAE 软件“网格 → 装配 → 求解 → 后处理”与 Python 代码的对应关系；
2. 建立独立的 conda 环境，并让 Jupyter 内核指向它；
3. 运行 `env_check.py`，读懂 `PASS` / `FAIL`；
4. 在 Jupyter 中改参数、重新运行并解释云图变化；
5. 遇到路径、内核和字体三类故障时，使用证据定位原因。

需要准备：高等数学（偏导数、通量、高斯公式）、线性代数（`Ax=b`）、大学物理中的热传导与波动、基础 Python。**不要求**深度学习、有限元理论或泛函分析基础。

## 2. 先修自测（7 题）

先独立作答，再对照答案。自测不计分，作用是决定补修方向。

| 题目 | 问题 | 答案 |
|---|---|---|
| Q1 高数 | 若 `u=e^(-t) sin(πx)`，则 `∂²u/∂x²`？ | B：`-π² e^(-t) sin(πx)` |
| Q2 高数 | 高斯散度公式中的通量等于什么？ | A：`∭ ∇·F dV` |
| Q3 线代 | 可逆 `A` 的 `Ax=b` 解？ | B：`x=A⁻¹b`（实际计算优先调用求解器） |
| Q4 线代 | `diag(2,3)` 的特征值？ | C：2 和 3 |
| Q5 物理 | `q=-k∇T` 中负号的含义？ | B：热量从高温流向低温 |
| Q6 NumPy | `np.ones((3,4)).shape`？ | B：`(3,4)` |
| Q7 Python | `sum(i for i in range(4))`？ | B：6 |

答错的 Python/NumPy 题对照官方教程和 NumPy Quickstart；数学物理题回到自己的教材对应章节。

## 3. 失效驱动：先看一个熟悉的崩溃时刻

商业软件里常见的 `Non-linear solver did not converge`、`NaN or Inf found in solution vector` 只告诉你“失败了”，不告诉你是模型、边界条件、步长还是迭代过程出了问题。请先写下自己的判断，再运行下一格，让 Python 亲手“算崩”一次。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def explicit_heat(r, nx=41, nsteps=200):
    """显式差分解 u_t = u_xx；r = dt/dx^2，两端 u = 0。"""
    x = np.linspace(0, 1, nx)
    u = np.sin(np.pi * x)
    peak = [np.abs(u).max()]
    for _ in range(nsteps):
        u[1:-1] = u[1:-1] + r * (u[2:] - 2 * u[1:-1] + u[:-2])
        peak.append(np.abs(u).max())
    return np.array(peak)


nsteps = 200  # 改成 4000，r=0.6 会进一步溢出为 inf / nan
plt.figure(figsize=(6, 3.8))
for r in (0.4, 0.5, 0.6):
    peak = explicit_heat(r, nsteps=nsteps)
    print(f"r = {r}: 第 {nsteps} 步时 max|u| = {peak[-1]:.3e}")
    plt.semilogy(peak, label=f"r = {r}")
plt.axhline(1.0, color="gray", ls=":", lw=1)
plt.xlabel("time step")
plt.ylabel("max |u|")
plt.title("Explicit scheme: stable vs blow-up")
plt.legend()
plt.show()

### 观察与解释

预期现象：`r=0.4`、`0.5` 衰减，`r=0.6` 爆炸（约第 113 步越过 1，200 步约 `1e12`）。稳定性条件是扩散问题的 `r≤1/2`，不是波动方程的 CFL。即便初值是光滑的 `sin(πx)`，约 `1e-16` 的舍入误差也会播种最高频模态；在 `r=0.6` 时该模态每步按 `|1-4r|=1.4` 倍放大。完整分析在 M2.2。

## 4. 失效地图 F1–F5

| 编号 | 现象 | 常见病因 | 课程回答 |
|---|---|---|---|
| F1 | `Non-linear solver did not converge` | 初值、载荷步、切线刚度；PINN 优化困难 | M3.6、M4.3 |
| F2 | `NaN / Inf` | 扩散稳定性或 CFL 违反；单位错误；除零 | M2.2–M2.4 |
| F3 | 矩阵奇异 / 刚体位移 | 约束不足、纯 Neumann、过约束 | M1.2、M3.5 |
| F4 | 算完了但结果不合理 | 模型、单位、边界错误；未验证、未收敛 | 全课：精确解、收敛阶、守恒/能量 |
| F5 | AI 结果看着对但不可靠 | 谱偏差、损失权重失衡、分布外失效 | M4.1、M4.3、M5.1 |

共同地基是线性代数：非线性迭代的每一步都要解形如 `J Δu = -R` 的线性系统。

## 5. 课程地图与黑盒拆解

三大锚点方程沿五条方法轨道推进：**M1 解析基准 → M2 FDM/FVM → M3 FEM → M4 PINN → M5 算子学习**。每一圈都要配经典基线、验证三件套（精确解/收敛阶/守恒或能量）和失效卡。

商业软件的五步与 Python 对应关系：

1. 区域离散：`meshgrid` 或三角形/四边形网格；
2. 方程离散：微分算子离散权重，或弱形式与单元积分；
3. 装配：FDM 的差分系数矩阵 `scipy.sparse.diags`，FEM 的逐单元刚度矩阵 `K`；
4. 边界条件：修改自由度对应的行与右端项；
5. 求解与后处理：`spsolve`、整形、`contourf`。

规则矩形上 FDM 直观高效；L 形域、带孔板和异形散热片更适合在 M3 引入 FEM。FEniCSx 是 T2 可选工具，不进入 T0 必修路径。

## 6. 解析解基准：第一张二维温度场云图

底边和两侧为 `0 °C`，顶边按 `T0·sin(πx)` 加热。它是解析公式，不是数值解；到 M2 将用差分矩阵逼近它，并把它作为标准答案。先运行一次，再把 `T0` 改成 `200` 重跑。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T0 = 100  # 顶边最高温度 (°C)，试试改成 200
x = np.linspace(0, 1, 50)
X, Y = np.meshgrid(x, x)
T = T0 * np.sin(np.pi * X) * np.sinh(np.pi * Y) / np.sinh(np.pi)

plt.figure(figsize=(5.5, 4))
plt.contourf(X, Y, T, levels=20, cmap="inferno")
plt.colorbar(label="Temperature (°C)")
plt.title("2D Steady Heat Conduction (analytical)")
plt.show()

## 7. 课前环境清单

推荐 Anaconda/Miniconda + Python 3.12，项目放在纯英文路径。安装请课前完成：

```bash
conda create -n sciml_pde python=3.12 -y
conda activate sciml_pde
pip install numpy scipy matplotlib sympy jupyter pytest
# PyTorch 使用 CPU 版即可（M4 起需要）
python -m ipykernel install --user --name sciml_pde --display-name "Python (SciML)"
```

Windows 使用 Anaconda Prompt；macOS/Linux 使用终端。网页的 Pyodide 可作为 M0–M3 的零门槛备选，但没有 PyTorch。

## 8. Jupyter 速成

- `Esc`：命令模式（蓝框）；`Enter`：编辑模式（绿框）；
- `Shift+Enter`：运行当前单元并跳到下一格；
- 命令模式下 `B/A`：在下方/上方插入单元格；
- `Tab` 自动补全，`Shift+Tab` 查看函数说明；
- 右上角内核必须是 **Python (SciML)**。

如果终端里已经安装了包、Notebook 却找不到，优先检查 `sys.executable`，不要盲目重复安装。

## 9. 环境自检：真实检测与演示模式

下面调用本单元的权威源码 `code/env_check.py`。真实检测中的 PyTorch `FAIL`（若浏览器环境运行）是预期结果；演示模式的 `SIMULATED` 只用于说明全绿长什么样，不能替代真实安装验证。

In [ ]:
from pathlib import Path
import sys


search_roots = (Path.cwd(), *Path.cwd().parents)
code_candidates = [parent / "code" for parent in search_roots]
for root in search_roots:
    code_candidates.extend(root.glob("00_M0_起点/*/code"))
CODE_DIR = next((path.resolve() for path in code_candidates if (path / "env_check.py").is_file()), None)
if CODE_DIR is None:
    raise FileNotFoundError("找不到 code/env_check.py，请从本单元 notebooks 目录或项目根目录运行")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import env_check as S

ASSET_DIR = CODE_DIR.parent / "assets"
S.check_environment(save_path=str(ASSET_DIR / "env_status.png"))

In [ ]:
from pathlib import Path
import sys

search_roots = (Path.cwd(), *Path.cwd().parents)
code_candidates = [parent / "code" for parent in search_roots]
for root in search_roots:
    code_candidates.extend(root.glob("00_M0_起点/*/code"))
CODE_DIR = next((path.resolve() for path in code_candidates if (path / "env_check.py").is_file()), None)
if CODE_DIR is None:
    raise FileNotFoundError("找不到 code/env_check.py")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import env_check as S

# 仅演示：把 torch 标记为 SIMULATED，不代表真实环境已安装。
S.check_environment(simulate={"torch"}, save_path=str(CODE_DIR.parent / "assets" / "env_status_demo.png"))

## 10. 三大暗坑与排错口诀

1. **路径含中文/特殊字符**：底层工具可能按本地编码解析失败；项目和 conda 目录放纯英文路径。
2. **虚拟环境混淆**：`pip list` 有包不代表 Jupyter 内核指向同一个 Python；运行 `import sys; print(sys.executable)` 核对。
3. **中文变方块**：Matplotlib 的字体回退列表没有中文字形；网页环境统一使用英文图标签。

跨平台字体配置示例：

```python
import matplotlib.pyplot as plt
plt.rcParams["font.sans-serif"] = [
    "SimHei", "Microsoft YaHei",       # Windows
    "PingFang SC",                      # macOS
    "Noto Sans CJK SC", "WenQuanYi Micro Hei",  # Linux
    "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
```

## 11. 诊断挑战（代码面试）

先看证据，再猜原因，最后才动手：

| 场景 | 关键证据 | 最可能原因与处理 |
|---|---|---|
| `pip list` 有 torch，Jupyter 导入失败 | `sys.executable` 指向 `anaconda3/python.exe` 而不是 `envs/sciml_pde` | 内核指向 base；在激活环境中注册 `Python (SciML)` 并切换内核 |
| 中文用户名触发 `UnicodeDecodeError` | 报错路径含中文，且有编码解码失败 | 路径编码问题；迁移到纯英文路径 |
| 标题显示方块 | `Glyph missing from font` | 配置含中文字体的回退列表，Linux 先安装字体 |
| `X, Y = np.meshgrid(x)` 解包失败 | `expected 2, got 1` | 传入 `x, y` 两个数组：`X, Y = np.meshgrid(x, y)` |

共同套路：看报错最后一行、看路径、看警告原文。AI 可以帮你写代码，但判断“为什么失败”是自己的能力。

## 12. Lab 1：环境初始化

这是网页 v2 的“实验台 1”。它只打印版本和绘图后端，不依赖前面任何变量。

In [ ]:
import platform
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print("=" * 56)
print("  《M0.1 交互式实验工作台》已就绪")
print("=" * 56)
print("Python       :", platform.python_version())
print("NumPy 版本   :", np.__version__)
print("Matplotlib   :", matplotlib.__version__)
print("绘图后端     :", matplotlib.get_backend())
print()
print("本地环境：SciPy / SymPy / PyTorch 是否可用，请运行上面的 env_check 单元")

## 13. POE：先预测，再运行，再解释

**运行下一格前，先填写预测。** 预测不要求一次正确，关键是接受云图证据检验。

| 参数变化 | 我的预测（运行前填写） | 运行后的观察与解释 |
|---|---|---|
| `T0` 增大 |  |  |
| `n` 增大 |  |  |
| `N` 加密 |  |  |

提交时回答：预测是否正确？哪一条观察推翻或修正了原先的直觉？

## 14. Lab 2：改参数，看云图怎么变

先运行一次，再修改 `T0`、`n`、`N` 重跑。预期：`T0` 只改变幅值；`n` 越大，高温区向板内渗透越浅；`N` 只改变解析图的显示精细度，不改变物理解。到了 M2 的数值解，`N` 才会影响离散精度。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T0 = 150  # 顶边最高温度 (°C)
n = 4     # 顶边正弦的半波个数：1, 2, 3 ...
N = 80    # 每个方向的网格点数

x = np.linspace(0, 1, N)
X, Y = np.meshgrid(x, x)
T = T0 * np.sin(n * np.pi * X) * np.sinh(n * np.pi * Y) / np.sinh(n * np.pi)

plt.figure(figsize=(5.5, 4))
plt.contourf(X, Y, T, levels=20, cmap="inferno")
plt.colorbar(label="Temperature (°C)")
plt.title(f"n = {n}, T0 = {T0}")
plt.show()
print(f"最高温度 = {T.max():.1f} °C")

## 15. Lab 3：向量化到底快多少？

不要背网页预置的毫秒数；机器不同，耗时不同。下面对一百万个数做点积，用纯 Python 循环与 `np.dot` 对照，并重复向量化版本 50 次。

In [ ]:
import time
import numpy as np

N = 1_000_000
a = np.random.rand(N)
b = np.random.rand(N)

t0 = time.perf_counter()
s = 0.0
for i in range(N):
    s += a[i] * b[i]
t_loop = time.perf_counter() - t0

reps = 50
t0 = time.perf_counter()
for _ in range(reps):
    s_np = np.dot(a, b)
t_np = (time.perf_counter() - t0) / reps

print(f"for 循环 : {t_loop * 1000:9.1f} ms")
print(f"np.dot   : {t_np * 1000:9.3f} ms")
print(f"加速比   : {t_loop / t_np:9.0f} 倍（因机器而异，以你测到的为准）")
print(f"两种结果相差 {abs(s - s_np):.2e}（可能恰好为 0，也可能是极小的浮点误差）")

## 16. 课后 Checklist 与下讲预告

- [ ] 激活 `sciml_pde` 环境；
- [ ] 运行 `python code/env_check.py`，得到 5 个 `[PASS]` 并生成 `assets/env_status.png`；
- [ ] 启动 Jupyter，打开本 Notebook，内核选 `Python (SciML)`；
- [ ] 依次运行单元，复现解析温度场云图；
- [ ] 修改 `T0`、`n`，记录云图变化并写一句解释；
- [ ] 在本地重新测向量化加速比，与网页结果比较。

下一讲是 **M1.1 控制体守恒与三大方程**：从一个微元小方盒推导热传导、扩散和对流方程。今天看到的 `r=0.6` 爆炸，会在 M2.2 由你亲手复现并解释。